# Imports

In [ ]:
# from logit_model import *
import statsmodels.api as sm
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from statsmodels.stats.outliers_influence import variance_inflation_factor




In [ ]:
import seaborn as sns
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)


In [ ]:
from package_files.benefits_defns import *

In [ ]:
from package_files.logit_model import *

In [ ]:
# reload logit_model
from importlib import reload
reload(sys.modules['package_files.logit_model'])

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
path = '../data/small_samples/2024_salary_sample.parquet.gzip'

In [ ]:
if path[-3:] == 'csv:':
    even_sample = pd.read_csv(path)
elif path[-3:] == 'zip':
    even_sample = pd.read_parquet(path)

In [ ]:
even_sample

In [ ]:
even_sample[even_sample['YEAR'] == 2018]['BODY']

In [ ]:
even_sample[even_sample['YEAR'] == 2018].groupby('WLB').size()

In [ ]:
region = 'STATE_NAME'
industry = 'NAICS_2022_2_NAME'
education = 'MIN_EDULEVELS_NAME'
year = 'YEAR'
occupation = 'SOC_2021_2_NAME'
experience = 'EXPERIENCE_BUCKET'

# Define Function

In [ ]:
def run_logit_model(data, dependent, cat_controls=[], cont_controls = [], binary_vars = [], predictor = 'Has AI Skills', ref_category = None, get_vif = False):
    """
    Runs a logistic regression model to predict work-from-home dependent status based on the specified predictor
    and control variables.

    Parameters:
    -----------
    controls : list of str
        A list of column names to be used as control variables. These variables will be converted to categorical
        variables and dummy variables will be created for them.
    predictor : str, default='Has AI Skills'
        The column name of the primary predictor variable.
    data : pd.DataFrame, default=data
        The DataFrame containing the data.
    ref_category : dict, optional
        A dictionary specifying the reference category for each control variable where keys are column names
        and values are the reference categories.

    Returns:
    --------
    logit_model : statsmodels.discrete.discrete_model.BinaryResultsWrapper
        The fitted logistic regression model.

    Example:
    --------
    >>> controls = ['region_column_name', 'industry_column_name']
    >>> model = run_wfh_logit(controls, predictor='Has AI Skills', data=data)
    >>> print(model.summary())
    """
    X = data[[predictor]]
    
    if binary_vars:
        for var in binary_vars:
            X = pd.concat([X, data[[var]]], axis = 1)
    
    if cont_controls:
        for cc in cont_controls:
            X = pd.concat([X, data[[cc]]], axis = 1)
    
    if cat_controls:
        for c in cat_controls:    
            # Create categorical variables 
            data[c] = data[c].astype('category')
            
            # Create dummy variables with specified reference category
            if ref_category and c in ref_category:
                # Ensure the reference category is present in the data
                if ref_category[c] in data[c].cat.categories:
                    data[c] = data[c].cat.reorder_categories(
                        [ref_category[c]] + [cat for cat in data[c].cat.categories if cat != ref_category[c]],
                        ordered=True
                    )
                    dummies = pd.get_dummies(data[c], drop_first=True)
            # Create dummy variables
                else:
                    raise ValueError(f"Reference category '{ref_category[c]}' not found in column '{c}'")
            else:
                dummies = pd.get_dummies(data[c], drop_first=True)
            X = pd.concat([X, dummies], axis = 1)

    X = X.apply(pd.to_numeric, errors='coerce')

    y = data[dependent]
    data = pd.concat([X, y], axis=1).dropna()
    X = data.drop(columns = dependent)
    y = data[dependent]
    X = sm.add_constant(X)
    y = pd.to_numeric(y, errors='coerce')

    X.columns = X.columns.astype(str)

    X = X.astype(float)
    y = y.astype(float)
    # print(X)
    # print(X.corr())
    
    if get_vif:
        print("VIF Results")
        vif_data = []
        for i in tqdm(range(X.shape[1]), desc="Calculating VIF"):
            vif = variance_inflation_factor(X.values, i)
            vif_data.append(vif)
        vif = pd.DataFrame()
        vif["features"] = X.columns
        vif["VIF"] = vif_data
        print(vif)
        print("Correlation Matrix")
        # create new df combine X and y for correlation matrix
        dummy_df = pd.concat([X, y], axis=1)
        dummy_corr_matrix = dummy_df.corr()
        # print any correlations above 0.7
        print(dummy_corr_matrix[dummy_corr_matrix > 0.6])
        
    # if error, continue to next model
    try:
        logit_model = sm.Logit(y, X).fit()
    except Exception as error:
        # print error
        print(error)
        return "Error"

    # Print the summary of the model
    print(logit_model.summary())
    # print(summary_col(logit_model, stars=True, float_format='%0.2f'))
    return logit_model

In [ ]:
def get_dummy_df(data, dependent, cat_controls=[], cont_controls = [], binary_vars = [], predictor = 'Has AI Skills', ref_category = None):
    X = data[[predictor]]
    
    if binary_vars:
        for var in binary_vars:
            X = pd.concat([X, data[[var]]], axis = 1)
    
    if cont_controls:
        for cc in cont_controls:
            X = pd.concat([X, data[[cc]]], axis = 1)
    
    if cat_controls:
        for c in cat_controls:    
            # Create categorical variables 
            data[c] = data[c].astype('category')
            
            # Create dummy variables with specified reference category
            if ref_category and c in ref_category:
                # Ensure the reference category is present in the data
                if ref_category[c] in data[c].cat.categories:
                    data[c] = data[c].cat.reorder_categories(
                        [ref_category[c]] + [cat for cat in data[c].cat.categories if cat != ref_category[c]],
                        ordered=True
                    )
                    dummies = pd.get_dummies(data[c], drop_first=True)
            # Create dummy variables
                else:
                    raise ValueError(f"Reference category '{ref_category[c]}' not found in column '{c}'")
            else:
                dummies = pd.get_dummies(data[c], drop_first=True)
            X = pd.concat([X, dummies], axis = 1)

    X = X.apply(pd.to_numeric, errors='coerce')

    y = data[dependent]
    data = pd.concat([X, y], axis=1).dropna()
    X = data.drop(columns = dependent)
    y = data[dependent]
    X = sm.add_constant(X)
    y = pd.to_numeric(y, errors='coerce')

    X.columns = X.columns.astype(str)

    X = X.astype(float)
    y = y.astype(float)
    dummy_df = pd.concat([X, y], axis=1)
    return dummy_df
    

# Models with Only Year and Occupation Control

In [ ]:
benefits = ['CAREER_DEV', 'EDU_ASSISTANCE','WLB', 'WELLBEING', 'HEALTH_WELLBEING','RECOGNITION', 'FAMILY', 'PARENTAL_LEAVE','CULTURE']


In [ ]:
benefit_models = []
for benefit in benefits:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, occupation], get_vif = True)
    benefit_models.append(model)

In [ ]:
# benefit_models = []
# for benefit in benefits:
#     print(benefit)
#     print('--'*100)
#     model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, occupation])
#     benefit_models.append(model)

In [ ]:
# # export benefit_models to pickle
# import pickle
# if path[-3:] == 'csv:':
#     with open('../exports/benefit_models.pkl', 'wb') as f:
#         pickle.dump(benefit_models, f)
# elif path == '../data/salary_sample_body_benefits.parquet.gzip'
#     with open('../exports/benefit_models_fromlarge.pkl', 'wb') as f:
#         pickle.dump(benefit_models, f)

# With Individual Controls

In [ ]:
benefit_models_2 = []
for benefit in benefits:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, occupation, education],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = True)
    benefit_models_2.append(model)

In [ ]:
benefit_models_2

In [ ]:
pd.set_option('display.max_rows', 100)

In [ ]:
even_sample.groupby([occupation, education]).size()

In [ ]:
even_sample.groupby([occupation, experience]).size()

In [ ]:
even_sample.groupby(['AI ROLE', education]).size()

In [ ]:
even_sample.groupby(['AI ROLE', experience]).size()

In [ ]:
# import pickle
# if path[-3:] == 'csv:':
#     with open('../exports/benefit_models_2.pkl', 'wb') as f:
#         pickle.dump(benefit_models_2, f)
# elif path == '../data/salary_sample_body_benefits.parquet.gzip'
#     with open('../exports/benefit_models_fromlarge_2.pkl', 'wb') as f:
#         pickle.dump(benefit_models_2, f)

# With Salary Control

In [ ]:
benefit_models_3 = []
for benefit in benefits:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, occupation, education, experience], cont_controls = ['LOG_SALARY'], ref_category = {education: "No Education Listed", experience: 'None Listed'})
    benefit_models_3.append(model)

In [ ]:
benefit_models_3

# Troubleshooting

## Check for Multicollinearity

In [ ]:
dummy_df = get_dummy_df(even_sample, dependent = 'PARENTAL_LEAVE', predictor='AI ROLE', cat_controls = [year, occupation, education, experience], cont_controls = ['LOG_SALARY'], ref_category = {education: "No Education Listed", experience: 'None Listed'})

In [ ]:
dummy_df

In [ ]:
# get correlations > 0.5
dummy_corr_matrix = dummy_df.corr()
dummy_corr_matrix[(dummy_corr_matrix > 0.4) | (dummy_corr_matrix < -0.4)].stack().reset_index()



In [ ]:
# Filter correlations greater than 0.5 or less than -0.5, excluding diagonal (correlation of a variable with itself)
high_corr = dummy_corr_matrix[(dummy_corr_matrix > 0.5) | (dummy_corr_matrix < -0.5)].stack().reset_index()

# Renaming columns for better readability
high_corr.columns = ['Variable 1', 'Variable 2', 'Correlation']

# Remove self-correlations (where Variable 1 and Variable 2 are the same)
high_corr = high_corr[high_corr['Variable 1'] != high_corr['Variable 2']]

# Print the high correlations
high_corr

## Variation

In [ ]:
even_sample[benefits]

In [ ]:
for benefit in benefits:
    print(even_sample[benefit].sum())

In [ ]:
benefits3_labels

In [ ]:

    
# plot a clustered bar chart for all benefits
benefit_counts = even_sample[benefits3].sum()
benefit_counts.plot(kind='bar', figsize=(12, 6))
# rotate x labels
# set labels as benefits3_labels
plt.xticks(ticks=range(len(benefits3_labels)),labels=benefits3_labels)
plt.xticks(rotation=45, ha='right')



In [ ]:
even_sample['PARENTAL_LEAVE'].value_counts()

# Plot Coefficients

In [ ]:
coefficients = []
errors = []
pvalues = []
for model in benefit_models:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
    except:
        coef = None
        err = None
        pvalue = None

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)

# Creating DataFrame
results = {
    'Label': benefits,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues
}

results_df = pd.DataFrame(results)



In [ ]:
results_df

In [ ]:
results_df['Model Iteration'] = 'Baseline'

In [ ]:
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))
# Define colors for the 9 AI skills categories
# colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#999999']
cmap = plt.get_cmap("tab10")
colors = [cmap(i) for i in range(len(results_df))]
# Plot each AI skill category
for i, row in results_df.iterrows():
    ax.errorbar(
        i, row['Coefficient'], 
        yerr=row['Error'], 
        fmt='o', color=colors[i], label=row['Label'], markersize = 10
    )

# Customize plot
ax.set_xticks(range(len(results_df)))
ax.set_xticklabels(results_df['Label'], rotation=45, ha='right', fontsize = 16)
# ax.set_title('Coefficients for 9 AI Skills Categories')
# ax.set_xlabel('AI Skills Categories', fontsize = 18)
ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize = 16)
ax.axhline(0, color='grey', linewidth=0.8)
# ax.axhline(average_coefficient, color='grey', linestyle='--', linewidth=1, label='Average Coefficient')

# Increase the size of y-ticks
ax.tick_params(axis='y', labelsize=16)
ax.set_title('AI Role Log-Odds Coefficients with Occupation and Year Fixed Effects')
ax.set_ylim(-0.4, 1.0)
# Add legend
# ax.legend(title='AI Skills', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize = 16, title_fontsize = 16)

plt.tight_layout() 
plt.show()

In [ ]:
coefficients = []
errors = []
pvalues = []
for model in benefit_models_2:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
    except:
        coef = None
        err = None
        pvalue = None

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)

# Creating DataFrame
results_2 = {
    'Label': benefits,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues
}

results_df_2 = pd.DataFrame(results_2)



In [ ]:
results_df_2

In [ ]:
results_df_2['Model Iteration'] = 'Individual Controls'

In [ ]:
coefficients = []
errors = []
pvalues = []
for model in benefit_models_3:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
    except:
        coef = None
        err = None
        pvalue = None

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)

# Creating DataFrame
results_3 = {
    'Label': benefits,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues
}

results_df_3 = pd.DataFrame(results_3)



In [ ]:
results_df_3['Model Iteration'] = 'With Salary'

In [ ]:
df = pd.concat([results_df, results_df_2, results_df_3])

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each model
colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
# Calculate the 95% confidence intervals
df['Lower_CI'] = df['Coefficient'] - 1.96 * df['Error']
df['Upper_CI'] = df['Coefficient'] + 1.96 * df['Error']

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))

# benefits = benefits
model_iterations = df['Model Iteration'].unique()
markers = ['o', 's', '^']  # Different markers for model iterations
positions = []
current_pos = 0

# Store legend handles and labels to avoid duplicates
handles, labels = [], []

for benefit in benefits:
    benefit_data = df[df['Label'] == benefit]
    benefit_positions = []
    for i, model in enumerate(model_iterations):
        model_data = benefit_data[benefit_data['Model Iteration'] == model]
        if not model_data.empty:
            pos = current_pos + i * 0.2  # Adjust spacing between model_iterations within the same benefit
            handle = ax.errorbar(
                pos, model_data['Coefficient'].values, 
                yerr=[model_data['Coefficient'].values - model_data['Lower_CI'].values, 
                      model_data['Upper_CI'].values - model_data['Coefficient'].values], 
                fmt=markers[i], color=colors[i], label=model if benefit == benefits[0] else ""
            )
            benefit_positions.append(pos)
            if benefit == benefits[0]:  # Add handles and labels only for the first benefit to avoid duplicates
                handles.append(handle)
                labels.append(model)
    positions.extend(benefit_positions)
    current_pos += len(model_iterations) + 1  # Add more space between different benefits

# Customize plot
benefit_ticks = [(positions[i * len(model_iterations)] + positions[(i + 1) * len(model_iterations) - 1]) / 2 for i in range(len(benefits))]
ax.set_xticks(benefit_ticks)
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(benefits3_labels, rotation=45, fontsize = 14, ha='right')
# ax.set_title('AI Skill Coefficients by benefit and Occupation')
ax.set_xlabel('Benefit', fontsize=16)
ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize=16)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend(handles, labels, title='Model', bbox_to_anchor=(0, 1), loc = 'upper left', fontsize = 12, title_fontsize = 12)
# ax.axhline(average_coefficient, color='black', linestyle='--', linewidth=1, label='Average Coefficient')
plt.title('Models with Occupation and Year Fixed Effects')
plt.tight_layout()
plt.savefig('../figures/model_coefficients_plot_3.png')
plt.show()


# Models with Industry

In [ ]:
industry_counts = even_sample['NAICS_2022_2_NAME'].value_counts()
small_industries = industry_counts[industry_counts < 30].index  # Set a threshold (e.g., 10 observations)
even_sample['NAICS_2022_2_NAME'] = even_sample['NAICS_2022_2_NAME'].replace(small_industries, 'Other')

In [ ]:
benefits4

In [ ]:
even_sample_wham = pd.read_parquet('../data/small_samples/salary_wham_sample_20k.parquet.gzip')

In [ ]:
even_sample_wham.groupby('YEAR').size()

In [ ]:
even_sample.groupby('YEAR').size()

In [ ]:
benefit_models_industry = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    if (benefit == 'wfh_wham' or benefit == 'PARENTAL_LEAVE'):
        model = run_logit_model(even_sample_wham, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry], get_vif=True)
    else:
        model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry], get_vif = True)
    benefit_models_industry.append(model)

In [ ]:
benefit_models_industry[0].converged

## Diagnostics

In [ ]:
even_sample[industry].value_counts()

In [ ]:
even_sample[even_sample['PARENTAL_LEAVE']==1].groupby([year]).size()

In [ ]:
even_sample_wham[even_sample_wham['PARENTAL_LEAVE']==1].groupby([year]).size()

In [ ]:
even_sample.groupby('YEAR').size()

In [ ]:
even_sample.groupby([industry, 'HEALTH_WELLBEING']).size()

In [ ]:
even_sample.groupby(['YEAR', industry, 'PARENTAL_LEAVE']).size().reset_index()

In [ ]:
pd.set_option('display.max_rows', 120)

In [ ]:
even_sample.groupby(['YEAR', industry]).size().sort_values().reset_index()

In [ ]:
run_logit_model(even_sample, dependent = 'PARENTAL_LEAVE', predictor='AI ROLE', cat_controls = [industry, 'YEAR'], get_vif = True)


In [ ]:
# benefit_models = []
# for benefit in benefits:
#     print(benefit)
#     print('--'*100)
#     model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, occupation])
#     benefit_models.append(model)

In [ ]:
# # export benefit_models to pickle
# import pickle
# if path[-3:] == 'csv:':
#     with open('../exports/benefit_models.pkl', 'wb') as f:
#         pickle.dump(benefit_models, f)
# elif path == '../data/salary_sample_body_benefits.parquet.gzip'
#     with open('../exports/benefit_models_fromlarge.pkl', 'wb') as f:
#         pickle.dump(benefit_models, f)

# With Individual Controls

In [ ]:
benefit_models_industry_2 = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    if (benefit == 'wfh_wham'or benefit == 'PARENTAL_LEAVE'):
        model = run_logit_model(even_sample_wham, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = False)
    else:
        model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = False)
    benefit_models_industry_2.append(model)

In [ ]:
benefit_models_industry_2

# With Salary Control

In [ ]:
benefit_models_industry_3 = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    if (benefit == 'wfh_wham' or benefit == 'PARENTAL_LEAVE'):
        model = run_logit_model(even_sample_wham, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience], cont_controls = ['LOG_SALARY'], ref_category = {education: "No Education Listed", experience: 'None Listed'})
    else:
        model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience], cont_controls = ['LOG_SALARY'], ref_category = {education: "No Education Listed", experience: 'None Listed'})
    benefit_models_industry_3.append(model)

# Plot Coefficients

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df = pd.DataFrame(results_industry)



In [ ]:
results_industry_df['Model Iteration'] = 'Baseline'

In [ ]:
results_industry_df

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_2:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_2 = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_2 = pd.DataFrame(results_industry_2)



In [ ]:
results_industry_df_2['Model Iteration'] = 'Individual Controls'

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_3:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_3 = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_3 = pd.DataFrame(results_industry_3)



In [ ]:
results_industry_df_3['Model Iteration'] = 'With Salary'

In [ ]:
industry_df = pd.concat([results_industry_df, results_industry_df_2, results_industry_df_3])

In [ ]:
industry_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each model
colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
# Calculate the 95% confidence intervals
industry_df['Lower_CI'] = industry_df['Coefficient'] - 1.96 * industry_df['Error']
industry_df['Upper_CI'] = industry_df['Coefficient'] + 1.96 * industry_df['Error']

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))

# benefits = benefits
model_iterations = industry_df['Model Iteration'].unique()
markers = ['o', 's', '^']  # Different markers for model iterations
positions = []
current_pos = 0

# Store legend handles and labels to avoid duplicates
handles, labels = [], []

for benefit in benefits4:
    benefit_data = industry_df[industry_df['Label'] == benefit]
    benefit_positions = []
    for i, model in enumerate(model_iterations):
        model_data = benefit_data[benefit_data['Model Iteration'] == model]
        if not model_data.empty:
            pos = current_pos + i * 0.2  # Adjust spacing between model_iterations within the same benefit
            handle = ax.errorbar(
                pos, model_data['Coefficient'].values, 
                yerr=[model_data['Coefficient'].values - model_data['Lower_CI'].values, 
                      model_data['Upper_CI'].values - model_data['Coefficient'].values], 
                fmt=markers[i], color=colors[i], label=model if benefit == benefits4[0] else ""
            )
            benefit_positions.append(pos)
            if benefit == benefits4[0]:  # Add handles and labels only for the first benefit to avoid duplicates
                handles.append(handle)
                labels.append(model)
            if not model_data['Converged'].values[0]:  # Assuming Converged is boolean
                ax.plot(pos, model_data['Coefficient'].values[0], 'rx', markersize=12, label='Did Not Converge')
        
    positions.extend(benefit_positions)
    current_pos += len(model_iterations) + 1  # Add more space between different benefits

# Customize plot
benefit_ticks = [(positions[i * len(model_iterations)] + positions[(i + 1) * len(model_iterations) - 1]) / 2 for i in range(len(benefits4))]
ax.set_xticks(benefit_ticks, labels = benefits4_labels)
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(benefits4_labels, rotation=45, fontsize = 14, ha='right')
# ax.set_title('AI Skill Coefficients by benefit and Occupation')
ax.set_xlabel(None)
ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize=16)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend(handles, labels, title='Model', bbox_to_anchor=(0, 1), loc = 'upper left', fontsize = 12, title_fontsize = 12)
# ax.axhline(average_coefficient, color='black', linestyle='--', linewidth=1, label='Average Coefficient')
plt.title('Models with Industry and Year Fixed Effects', fontsize = 16)
plt.tight_layout()
plt.savefig('../figures/model_coefficients_plot_industry_converged.png')
plt.show()


# Without Salary

In [ ]:
even_sample_nosal = pd.read_parquet('../../data/nosalary_sample_20k.parquet.gzip')

In [ ]:
even_sample_nosal = even_sample_nosal[even_sample_nosal['YEAR'] != 2018]

In [ ]:
even_sample_nosal.groupby('YEAR').size()

In [ ]:
benefit_models_industry = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    if (benefit == 'wfh_wham' or benefit == 'PARENTAL_LEAVE'):
        model = run_logit_model(even_sample_nosal, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry], get_vif=True)
    else:
        model = run_logit_model(even_sample_nosal, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry], get_vif = True)
    benefit_models_industry.append(model)

## With Individual Controls

In [ ]:
benefit_models_industry_2 = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    if (benefit == 'wfh_wham'or benefit == 'PARENTAL_LEAVE'):
        model = run_logit_model(even_sample_nosal, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = False)
    else:
        model = run_logit_model(even_sample_nosal, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = False)
    benefit_models_industry_2.append(model)

In [ ]:
benefit_models_industry_2

## Plot Coefficients

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df = pd.DataFrame(results_industry)



In [ ]:
results_industry_df['Model Iteration'] = 'Baseline'

In [ ]:
results_industry_df

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_2:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_2 = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_2 = pd.DataFrame(results_industry_2)



In [ ]:
results_industry_df_2['Model Iteration'] = 'Individual Controls'

In [ ]:
industry_df = pd.concat([results_industry_df, results_industry_df_2])

In [ ]:
industry_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each model
colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
# Calculate the 95% confidence intervals
industry_df['Lower_CI'] = industry_df['Coefficient'] - 1.96 * industry_df['Error']
industry_df['Upper_CI'] = industry_df['Coefficient'] + 1.96 * industry_df['Error']

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))

# benefits = benefits
model_iterations = industry_df['Model Iteration'].unique()
markers = ['o', 's', '^']  # Different markers for model iterations
positions = []
current_pos = 0

# Store legend handles and labels to avoid duplicates
handles, labels = [], []

for benefit in benefits4:
    benefit_data = industry_df[industry_df['Label'] == benefit]
    benefit_positions = []
    for i, model in enumerate(model_iterations):
        model_data = benefit_data[benefit_data['Model Iteration'] == model]
        if not model_data.empty:
            pos = current_pos + i * 0.2  # Adjust spacing between model_iterations within the same benefit
            handle = ax.errorbar(
                pos, model_data['Coefficient'].values, 
                yerr=[model_data['Coefficient'].values - model_data['Lower_CI'].values, 
                      model_data['Upper_CI'].values - model_data['Coefficient'].values], 
                fmt=markers[i], color=colors[i], label=model if benefit == benefits4[0] else ""
            )
            benefit_positions.append(pos)
            if benefit == benefits4[0]:  # Add handles and labels only for the first benefit to avoid duplicates
                handles.append(handle)
                labels.append(model)
            if not model_data['Converged'].values[0]:  # Assuming Converged is boolean
                ax.plot(pos, model_data['Coefficient'].values[0], 'rx', markersize=12, label='Did Not Converge')
        
    positions.extend(benefit_positions)
    current_pos += len(model_iterations) + 1  # Add more space between different benefits

# Customize plot
benefit_ticks = [(positions[i * len(model_iterations)] + positions[(i + 1) * len(model_iterations) - 1]) / 2 for i in range(len(benefits4))]
ax.set_xticks(benefit_ticks, labels = benefits4_labels)
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(benefits4_labels, rotation=45, fontsize = 14, ha='right')
# ax.set_title('AI Skill Coefficients by benefit and Occupation')
ax.set_xlabel(None)
ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize=16)
ax.set_ylim(-0.5, 1.75)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend(handles, labels, title='Model', bbox_to_anchor=(0, 1), loc = 'upper left', fontsize = 12, title_fontsize = 12)
# ax.axhline(average_coefficient, color='black', linestyle='--', linewidth=1, label='Average Coefficient')
plt.title('Models with Industry and Year Fixed Effects', fontsize = 16)
plt.tight_layout()
# plt.savefig('../figures/model_coefficients_plot_industry_converged.png')
plt.show()


In [ ]:
even_sample.columns

# Remote Keywords

In [ ]:
industry_counts = even_sample['NAICS_2022_2_NAME'].value_counts()
small_industries = industry_counts[industry_counts < 30].index  # Set a threshold (e.g., 10 observations)
even_sample['NAICS_2022_2_NAME'] = even_sample['NAICS_2022_2_NAME'].replace(small_industries, 'Other')

In [ ]:
benefits4

In [ ]:
benefits5 = ['EDU_ASSISTANCE',
 'PAID_LEAVE',
 'HEALTH_WELLBEING',
 'PARENTAL_LEAVE',
 'CULTURE',
 'wfh_wham',
 'Remote_KW']

In [ ]:
even_sample.head()

In [ ]:
benefit_models_industry = []
for benefit in benefits5:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry], get_vif = True)
    benefit_models_industry.append(model)

In [ ]:
benefit_models_industry_2 = []
for benefit in benefits5:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = False)
    benefit_models_industry_2.append(model)

In [ ]:
benefit_models_industry_3 = []
for benefit in benefits5:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience], cont_controls = ['LOG_SALARY'], ref_category = {education: "No Education Listed", experience: 'None Listed'})
    benefit_models_industry_3.append(model)

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry = {
    'Label': benefits5,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df = pd.DataFrame(results_industry)



In [ ]:
results_industry_df['Model Iteration'] = 'Baseline'

In [ ]:
results_industry_df

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_2:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_2 = {
    'Label': benefits5,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_2 = pd.DataFrame(results_industry_2)



In [ ]:
results_industry_df_2['Model Iteration'] = 'Individual Controls'

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_3:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_3 = {
    'Label': benefits5,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_3 = pd.DataFrame(results_industry_3)



In [ ]:
results_industry_df_3['Model Iteration'] = 'With Salary'

In [ ]:
industry_df = pd.concat([results_industry_df, results_industry_df_2, results_industry_df_3])

In [ ]:
industry_df

In [ ]:
benefits4_labels

In [ ]:

benefits5_labels = ['Tuition Assistance',
 'Paid Leave',
 'Health and Wellbeing',
 'Parental Leave',
 'Workplace Culture',
 'Remote Work', 'Remote Work Keywords']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each model
colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
# Calculate the 95% confidence intervals
industry_df['Lower_CI'] = industry_df['Coefficient'] - 1.96 * industry_df['Error']
industry_df['Upper_CI'] = industry_df['Coefficient'] + 1.96 * industry_df['Error']

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))

# benefits = benefits
model_iterations = industry_df['Model Iteration'].unique()
markers = ['o', 's', '^']  # Different markers for model iterations
positions = []
current_pos = 0

# Store legend handles and labels to avoid duplicates
handles, labels = [], []

for benefit in benefits5:
    benefit_data = industry_df[industry_df['Label'] == benefit]
    benefit_positions = []
    for i, model in enumerate(model_iterations):
        model_data = benefit_data[benefit_data['Model Iteration'] == model]
        if not model_data.empty:
            pos = current_pos + i * 0.2  # Adjust spacing between model_iterations within the same benefit
            handle = ax.errorbar(
                pos, model_data['Coefficient'].values, 
                yerr=[model_data['Coefficient'].values - model_data['Lower_CI'].values, 
                      model_data['Upper_CI'].values - model_data['Coefficient'].values], 
                fmt=markers[i], color=colors[i], label=model if benefit == benefits5[0] else ""
            )
            benefit_positions.append(pos)
            if benefit == benefits5[0]:  # Add handles and labels only for the first benefit to avoid duplicates
                handles.append(handle)
                labels.append(model)
            if not model_data['Converged'].values[0]:  # Assuming Converged is boolean
                ax.plot(pos, model_data['Coefficient'].values[0], 'rx', markersize=12, label='Did Not Converge')
        
    positions.extend(benefit_positions)
    current_pos += len(model_iterations) + 1  # Add more space between different benefits

# Customize plot
benefit_ticks = [(positions[i * len(model_iterations)] + positions[(i + 1) * len(model_iterations) - 1]) / 2 for i in range(len(benefits5))]
ax.set_xticks(benefit_ticks, labels = benefits5_labels)
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(benefits5_labels, rotation=45, fontsize = 14, ha='right')
# ax.set_title('AI Skill Coefficients by benefit and Occupation')
ax.set_xlabel(None)
ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize=16)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend(handles, labels, title='Model', bbox_to_anchor=(0, 1), loc = 'upper left', fontsize = 12, title_fontsize = 12)
# ax.axhline(average_coefficient, color='black', linestyle='--', linewidth=1, label='Average Coefficient')
plt.title('Models with Industry and Year Fixed Effects', fontsize = 16)
plt.tight_layout()
plt.savefig('../figures/model_coefficients_plot_industry_converged.png')
plt.show()


# 2024

# Experience

In [ ]:
# Define buckets for years of experience
bins = [-2, -1, 0, 2, 5, 10, 20, 100]
labels = ['Missing', '0 years', '1-2 years', '3-5 years', '6-10 years', '11-20 years', '21+ years']


In [ ]:
even_sample['EXPERIENCE_BUCKET'] = pd.cut(even_sample['MIN_YEARS_EXPERIENCE'], bins=bins, labels=labels, right=True)

In [ ]:
even_sample['EXPERIENCE_BUCKET']=even_sample['EXPERIENCE_BUCKET'].astype(str)

In [ ]:
# replace nan with 'None Listed'
even_sample['EXPERIENCE_BUCKET'] = even_sample['EXPERIENCE_BUCKET'].replace('nan', 'None Listed')

In [ ]:
even_sample['EXPERIENCE_BUCKET'].value_counts(dropna=False)

# Log Salary

In [ ]:
even_sample['LOG_SALARY'] = np.log(even_sample['SALARY'])

In [ ]:
even_sample.to_parquet('../data/small_samples/2024_salary_sample.parquet.gzip', compression='gzip')

In [ ]:
industry_counts = even_sample['NAICS_2022_2_NAME'].value_counts()
small_industries = industry_counts[industry_counts < 30].index  # Set a threshold (e.g., 10 observations)
even_sample['NAICS_2022_2_NAME'] = even_sample['NAICS_2022_2_NAME'].replace(small_industries, 'Other')

In [ ]:
benefits4

In [ ]:
benefit_models_industry = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry], get_vif = False)
    benefit_models_industry.append(model)

In [ ]:
benefit_models_industry_2 = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience],  ref_category = {education: "No Education Listed", experience: 'None Listed'}, get_vif = False)
    benefit_models_industry_2.append(model)

In [ ]:
benefit_models_industry_3 = []
for benefit in benefits4:
    print(benefit)
    print('--'*100)
    model = run_logit_model(even_sample, dependent = benefit, predictor='AI ROLE', cat_controls = [year, industry, education, experience], cont_controls = ['LOG_SALARY'], ref_category = {education: "No Education Listed", experience: 'None Listed'})
    benefit_models_industry_3.append(model)

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df = pd.DataFrame(results_industry)



In [ ]:
results_industry_df['Model Iteration'] = 'Baseline'

In [ ]:
results_industry_df

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_2:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_2 = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_2 = pd.DataFrame(results_industry_2)



In [ ]:
results_industry_df_2['Model Iteration'] = 'Individual Controls'

In [ ]:
coefficients = []
errors = []
pvalues = []
converged_list = []
observations = []
for model in benefit_models_industry_3:
    try:
        coef = model.params['AI ROLE']
        err = model.bse['AI ROLE']
        pvalue = model.pvalues['AI ROLE'].round(3)
        converged = model.converged 
        obs = model.nobs       
    except:
        coef = None
        err = None
        pvalue = None
        converged = None
        obs = None
        

    coefficients.append(coef)
    errors.append(err)
    pvalues.append(pvalue)
    converged_list.append(converged)
    observations.append(obs)

# Creating DataFrame
results_industry_3 = {
    'Label': benefits4,
    'Coefficient': coefficients,
    'Error': errors,
    'P-Value': pvalues,
    'Converged': converged_list, 
    'Observations': observations
}

results_industry_df_3 = pd.DataFrame(results_industry_3)



In [ ]:
results_industry_df_3['Model Iteration'] = 'With Salary'

In [ ]:
industry_df = pd.concat([results_industry_df, results_industry_df_2, results_industry_df_3])

In [ ]:
industry_df

In [ ]:
benefits4_labels

In [ ]:

benefits5_labels = ['Tuition Assistance',
 'Paid Leave',
 'Health and Wellbeing',
 'Parental Leave',
 'Workplace Culture',
 'Remote Work', 'Remote Work Keywords']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define colors for each model
colors = ['#E69F00', '#56B4E9', '#009E73', '#CC79A7']
# Calculate the 95% confidence intervals
industry_df['Lower_CI'] = industry_df['Coefficient'] - 1.96 * industry_df['Error']
industry_df['Upper_CI'] = industry_df['Coefficient'] + 1.96 * industry_df['Error']

# Plotting
fig, ax = plt.subplots(figsize=(12, 6))

# benefits = benefits
model_iterations = industry_df['Model Iteration'].unique()
markers = ['o', 's', '^']  # Different markers for model iterations
positions = []
current_pos = 0

# Store legend handles and labels to avoid duplicates
handles, labels = [], []

for benefit in benefits4:
    benefit_data = industry_df[industry_df['Label'] == benefit]
    benefit_positions = []
    for i, model in enumerate(model_iterations):
        model_data = benefit_data[benefit_data['Model Iteration'] == model]
        if not model_data.empty:
            pos = current_pos + i * 0.2  # Adjust spacing between model_iterations within the same benefit
            handle = ax.errorbar(
                pos, model_data['Coefficient'].values, 
                yerr=[model_data['Coefficient'].values - model_data['Lower_CI'].values, 
                      model_data['Upper_CI'].values - model_data['Coefficient'].values], 
                fmt=markers[i], color=colors[i], label=model if benefit == benefits4[0] else ""
            )
            benefit_positions.append(pos)
            if benefit == benefits4[0]:  # Add handles and labels only for the first benefit to avoid duplicates
                handles.append(handle)
                labels.append(model)
            if not model_data['Converged'].values[0]:  # Assuming Converged is boolean
                ax.plot(pos, model_data['Coefficient'].values[0], 'rx', markersize=12, label='Did Not Converge')
        
    positions.extend(benefit_positions)
    current_pos += len(model_iterations) + 1  # Add more space between different benefits

# Customize plot
benefit_ticks = [(positions[i * len(model_iterations)] + positions[(i + 1) * len(model_iterations) - 1]) / 2 for i in range(len(benefits4))]
ax.set_xticks(benefit_ticks, labels = benefits4_labels)
ax.tick_params(axis='y', labelsize=14)
ax.set_xticklabels(benefits4_labels, rotation=45, fontsize = 14, ha='right')
# ax.set_title('AI Skill Coefficients by benefit and Occupation')
ax.set_xlabel(None)
ax.set_ylabel('Log-Odds Coefficient (with 95% CI)', fontsize=16)
ax.axhline(0, color='grey', linewidth=0.8)
ax.legend(handles, labels, title='Model', bbox_to_anchor=(0, 1), loc = 'upper left', fontsize = 12, title_fontsize = 12)
# ax.axhline(average_coefficient, color='black', linestyle='--', linewidth=1, label='Average Coefficient')
# plt.title('Models with Industry and Year Fixed Effects', fontsize = 16)
plt.tight_layout()
plt.savefig('../figures/model_coefficients_plot_industry_converged.png')
plt.show()


# Tables

In [ ]:
from collections import defaultdict

In [ ]:
fixed_effects_variables = [year, industry]

fixed_effects_var_dict = {var: list(even_sample[var].unique()) for var in fixed_effects_variables}
cat_var_dict = {}

for category, values in fixed_effects_var_dict.items():
    # category_label = category.split('_')[0]  # Get the label part of the category (e.g., 'STATE' from 'STATE_NAME')
    for value in values:
        cat_var_dict[value] = category
cat_var_dict
fixed_effects_categories = list(fixed_effects_var_dict.values())
fixed_effects_categories = [item for sublist in fixed_effects_categories for item in sublist]

In [ ]:
fixed_effects_categories

In [ ]:
model_fixed_effects_dict

In [ ]:
def get_summary_table(models, export_name = 'table.tex'):
    """
    Generate a LaTeX table for a list of models with an indication of fixed effects variables.

    Parameters:
    - models: list of fitted model objects.
    - fixed_effects_variables: list of fixed effects variables.
    - filename: name of the output LaTeX file.
    """   
    # Assuming model1 and model2 are your fitted model objects
    # Create the Stargazer object with the models
    stargazer = Stargazer(models)
    # List of fixed effects variables
    # Automatically determine covariates to include (excluding fixed effects)
    
    cov_names = stargazer.cov_names
    if 'AI ROLE' in cov_names:
        cov_names.remove('AI ROLE')
        cov_names.insert(0, 'AI ROLE')
    stargazer.cov_names = cov_names
    
    
    fixed_effects_list = [cat for cat in fixed_effects_categories if cat in stargazer.cov_names]
    covariates_to_include = [covariate for covariate in stargazer.cov_names if covariate not in fixed_effects_list]
    
    # Remove unwanted covariates from stargazer
    for covariate in fixed_effects_list:
        if covariate in stargazer.cov_names:
            stargazer.cov_names.remove(covariate)

    
    # Add custom line for each fixed effect variable
    # stargazer.add_line("Fixed Effects")
    model_fixed_effects_dict = defaultdict(list)
    for i, model in enumerate(models): 
        model_fixed_effects_dict[i] = list(set([value for key, value in cat_var_dict.items() if key in model.model.exog_names]))
        
    fixed_effects_names = {'STATE_NAME': 'State',  'NAICS_2022_2_NAME': 'Industry', 'YEAR': 'Year', 'SOC_2021_2_NAME': 'Occupation'}    
    
    stargazer.add_line("\\textbf{Fixed Effects}", [""] * len(models))

    for variable in fixed_effects_variables:
        # fixed_effects = [value for key: value in cat_var_dict if key in stargazer.cov_names]
        variable_name = fixed_effects_names[variable]
        inclusion_list = ['Yes' if variable in model_fixed_effects_dict[model] else 'Yes' for model in model_fixed_effects_dict]
        stargazer.add_line(f"{variable_name}", inclusion_list)

    # Customize the Stargazer table (optional)
    # stargazer.title("Regression Results")
    # stargazer.custom_columns([f"Model {i+1}" for i in range(len(models))], [1] * len(models))

    stargazer.significance_levels([0.1, 0.05, 0.01])
    # stargazer.add_line("Observations", [len(model.model.endog) for model in models])

    # Render LaTeX table
    latex_table = stargazer.render_latex()
    print(latex_table)
    # export latex table
    with open(f'../exports/tables/{export_name}', 'w') as file:
        file.write(latex_table)
    
    display(HTML(stargazer.render_html()))
    # Print the LaTeX table to verify

In [ ]:
get_summary_table(model_progression, export_name = 'job_level_model.tex')

In [ ]:
model_progression = [models[0] for models in [benefit_models_industry, benefit_models_industry_2, benefit_models_industry_3]]

In [ ]:
model_progression

In [ ]:
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML

In [ ]:
stargazer = Stargazer(model_progression)
display(HTML(stargazer.render_html()))